# 03. Attention에서 접촉(contact)을 읽기 — ESM-2와 실제 구조 비교

**등급**: 🟢 Mac 로컬 (MPS/CPU) · **예상 시간**: 3~8분 · **다운로드**: ESM-2 8M~650M, PDB 3개(1UBQ·1PGA·1AKI, 합계 약 0.25 MB)

**질문**: 서열만 보고 MLM으로 학습한 ESM-2의 attention에는 **3D 접촉 정보**가 들어 있는가? 모델이 커지면 더 잘 읽히는가?

**정답**: 실제 구조(PDB)의 잔기쌍 접촉이다. Cβ(글리신은 Cα)–Cβ 거리가 8 Å 미만이면 접촉으로 본다.
**평가**: 서열 상에서 `MIN_SEP` 이상 떨어진 쌍 중 점수 상위 L/5개의 정밀도(precision@L/5)를 본다.

**비교 대상**
1. 무작위 예측 기대값(접촉 밀도) / 서열 거리 prior(가까운 쌍을 우선)
2. 모든 층·head의 attention을 평균하고 대칭화 + APC 보정한 점수 (비지도)
3. 모델에 포함된 **지도 학습된 contact head**(`predict_contacts`). 가장 강한 비교 대상이지만 접촉 라벨로 학습된 로지스틱 회귀가 들어 있다†

> 소형 단백질 3종이라 **개념 시연**이며 통계적 벤치마크가 아니다.

In [ ]:
import os
import urllib.request
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


def find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "SETUP.md").exists():
            return q
    return None


ROOT = find_root()
if ROOT is not None:
    os.environ.setdefault("HF_HOME", str(ROOT / ".cache" / "hf"))    # 원하면 지워 기본 HF 캐시를 쓴다
PDB_DIR = (ROOT / ".cache" / "pdb") if ROOT else Path("pdb_files")
PDB_DIR.mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import torch
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
from transformers import AutoTokenizer, EsmForMaskedLM
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE, "| pdb dir:", PDB_DIR)

## 1. 구조 불러오기와 정답 접촉
RCSB PDB에서 3개를 받는다 (출처: files.rcsb.org/download/<ID>.pdb). 표준 잔기 중 Cα가 있는 것만 쓰고, 서열은 같은 잔기들로 만든다.
잔기 번호에 **빈 구간(결측 잔기)**이 있으면 서열이 이어 붙여지므로 경고를 출력한다.

In [ ]:
PROTEINS = {"1UBQ": "A", "1PGA": "A", "1AKI": "A"}
for pid in PROTEINS:
    path = PDB_DIR / f"{pid}.pdb"
    if not path.exists():
        print("downloading", pid)
        urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pid}.pdb", path)


def load_structure(pid, chain_id):
    s = PDBParser(QUIET=True).get_structure(pid, PDB_DIR / f"{pid}.pdb")
    residues = [r for r in s[0][chain_id] if r.id[0] == " " and "CA" in r]
    seq = "".join(seq1(r.get_resname()) for r in residues)
    nums = [r.id[1] for r in residues]
    if any(b - a != 1 for a, b in zip(nums, nums[1:])):
        print(f"[경고] {pid}: 잔기 번호에 빈 구간이 있음 — 서열이 이어 붙여짐")
    def cb_coord(r):
        if r.get_resname() == "GLY":
            return r["CA"].coord                                    # 글리신만 Cα를 쓴다
        if "CB" in r:
            return r["CB"].coord
        print(f"[경고] {pid} {r.get_resname()}{r.id[1]}: Cβ 원자가 없어 Cα로 대체 (접촉 정답이 달라질 수 있음)")
        return r["CA"].coord

    cb = np.array([cb_coord(r) for r in residues])
    dist = np.linalg.norm(cb[:, None] - cb[None], axis=-1)
    return seq, dist


structs = {pid: load_structure(pid, ch) for pid, ch in PROTEINS.items()}
MIN_SEP = 12         # 서열 거리 ≥ 12인 쌍(중·장거리)만 평가. 작은 단백질이라 24 대신 12 사용
CUTOFF = 8.0
for pid, (seq, dist) in structs.items():
    L = len(seq)
    iu = np.triu_indices(L, k=MIN_SEP)
    print(f"{pid}: L={L}  접촉 쌍(<{CUTOFF:.0f}Å, 거리≥{MIN_SEP}): {(dist[iu] < CUTOFF).sum():4d} / {len(iu[0])}  → 밀도 {(dist[iu] < CUTOFF).mean():.3f}")

## 2. 평가 함수와 trivial baseline

In [ ]:
def precision_top(scores, dist, k=None, min_sep=MIN_SEP):
    L = scores.shape[0]
    k = k or max(1, L // 5)
    iu = np.triu_indices(L, k=min_sep)
    top = np.argsort(-scores[iu])[:k]
    return float((dist[iu][top] < CUTOFF).mean())


def sep_prior(L):
    i, j = np.indices((L, L))
    return -np.abs(i - j).astype(float)                      # 서열상 가까운 쌍일수록 높은 점수


for pid, (seq, dist) in structs.items():
    L = len(seq)
    iu = np.triu_indices(L, k=MIN_SEP)
    print(f"{pid}: random 기대 {(dist[iu] < CUTOFF).mean():.2f} | 서열 거리 prior {precision_top(sep_prior(L), dist):.2f}")

## 3. ESM-2의 attention과 contact head
attention 가중치를 얻으려면 `attn_implementation="eager"`로 로드해야 한다 (기본 SDPA 구현은 attention을 반환하지 않는다).
`<cls>`/`<eos>`에 해당하는 행·열은 잘라 낸다.

In [ ]:
def apc(x):
    """Average Product Correction: 위치별 합이 큰 배경 편향을 제거."""
    a1, a2, a12 = x.sum(-1, keepdims=True), x.sum(-2, keepdims=True), x.sum((-1, -2), keepdims=True)
    return x - a1 * a2 / a12


@torch.no_grad()
def contact_scores(tok, model, seq):
    enc = tok(seq, return_tensors="pt").to(DEVICE)
    out = model(**enc, output_attentions=True)
    att = torch.stack(out.attentions)[:, 0, :, 1:-1, 1:-1].float()           # (layers, heads, L, L), cls/eos 제거
    L = att.shape[-1]
    att_sym = att + att.transpose(-1, -2)                                    # 대칭화
    mean_sym = att_sym.mean((0, 1))
    eye = torch.eye(L, dtype=torch.bool, device=mean_sym.device)
    variants = {                                                             # 모든 층·head 평균에 보정을 단계적으로 적용 (비지도)
        "raw": att.mean((0, 1)).cpu().numpy(),                               # 보정 없음 (비대칭: 윗삼각만 쓰이므로 한 방향 attention만 반영)
        "sym": mean_sym.cpu().numpy(),                                       # 대칭화만
        "sym+APC": apc(mean_sym).cpu().numpy(),                              # 대칭화 + APC
        "sym+diag0+APC": apc(mean_sym.masked_fill(eye, 0)).cpu().numpy(),    # 자기 자신 attention(대각선)을 0으로 한 뒤 APC
    }
    per_head = apc(att_sym).cpu().numpy()                                    # (layers, heads, L, L)
    head = model.predict_contacts(enc["input_ids"], enc["attention_mask"])[0].float().cpu().numpy()   # 지도 학습된 contact head
    return variants, per_head, head


MODELS = {"8M": "facebook/esm2_t6_8M_UR50D", "35M": "facebook/esm2_t12_35M_UR50D",
          "150M": "facebook/esm2_t30_150M_UR50D", "650M": "facebook/esm2_t33_650M_UR50D"}
store, rows = {}, []
for mname, hf_id in MODELS.items():
    tok = AutoTokenizer.from_pretrained(hf_id)
    model = EsmForMaskedLM.from_pretrained(hf_id, attn_implementation="eager").to(DEVICE).eval()
    for pid, (seq, dist) in structs.items():
        variants, per_head, head = contact_scores(tok, model, seq)
        store[(mname, pid)] = (variants, per_head, head)
        rows.append(dict(model=mname, protein=pid, L=len(seq),
                         **{k: precision_top(v, dist) for k, v in variants.items()},
                         **{"contact head": precision_top(head, dist), "contact head (sep≥24)": precision_top(head, dist, min_sep=24)}))
    print(mname, "done | layers x heads:", per_head.shape[:2], "| contact head 출력 shape:", head.shape)
    del model
    if DEVICE.type == "mps":
        torch.mps.empty_cache()
res = pd.DataFrame(rows)

## 4. 결과 — precision@L/5 (서열 거리 ≥ 12)

In [ ]:
for col in ["sym+APC", "contact head"]:
    print(f"\n=== {col} ===")
    print(res.pivot(index="model", columns="protein", values=col).reindex(MODELS).round(2).assign(평균=lambda d: d.mean(1).round(2)).to_string())

### 보정의 효과 (3개 단백질 평균 precision@L/5)
같은 attention에 보정을 단계적으로 적용했을 때 점수가 어떻게 변하는지 본다. `raw`는 비대칭 attention의 윗삼각만 쓰므로 한 방향 정보만 반영한다.

In [ ]:
variant_cols = ["raw", "sym", "sym+APC", "sym+diag0+APC", "contact head"]
print(res.groupby("model")[variant_cols].mean().reindex(MODELS).round(2).to_string())

### 장거리 기준 (서열 거리 ≥ 24)
표준 접촉 벤치마크는 서열 거리에 따라 단거리(≥6)·중거리(≥12)·장거리(≥24)를 구분한다. 작은 단백질은 장거리 쌍이 적어 통계가 불안정하므로 쌍의 수를 함께 본다.

In [ ]:
for pid, (seq, dist) in structs.items():
    iu = np.triu_indices(len(seq), k=24)
    print(f"{pid}: 서열 거리 ≥ 24 쌍 {len(iu[0])}개 중 접촉 {(dist[iu] < CUTOFF).sum()}개, k = L//5 = {len(seq) // 5}")
print(res.pivot(index="model", columns="protein", values="contact head (sep≥24)").reindex(MODELS).round(2).assign(평균=lambda d: d.mean(1).round(2)).to_string())

## 5. head 하나를 고르면 좋아질까? — 누출 없는 선택
층·head마다 점수를 매겨 보면 일부 head가 접촉에 특화돼 있다. 그러나 **평가 단백질에서 가장 좋은 head를 고르면 낙관적**이다 (모듈 01).
여기서는 **leave-one-protein-out**: 평가 단백질을 뺀 나머지 단백질에서 평균이 가장 높은 head를 골라 평가 단백질에 적용한다.

In [ ]:
def head_precisions(per_head, dist):
    nl, nh = per_head.shape[:2]
    return np.array([[precision_top(per_head[l, h], dist) for h in range(nh)] for l in range(nl)])


out = []
for mname in ["35M", "150M"]:
    P = {pid: head_precisions(store[(mname, pid)][1], structs[pid][1]) for pid in structs}
    for pid in structs:
        others = np.mean([P[q] for q in structs if q != pid], axis=0)
        l, h = np.unravel_index(others.argmax(), others.shape)
        out.append(dict(model=mname, protein=pid, **{"선택 head (층,head)": f"({l},{h})", "선택 head 정밀도": P[pid][l, h],
                                                  "평가 단백질 내 최고 head(낙관적)": P[pid].max(), "낙관성 격차(최고−선택)": P[pid].max() - P[pid][l, h],
                                                  "평균 head": P[pid].mean()}))
gap = pd.DataFrame(out)
print(gap.round(2).to_string(index=False))
print("\n모델별 평균 낙관성 격차:", gap.groupby("model")["낙관성 격차(최고−선택)"].mean().round(2).to_dict(), "| 단백질 3개뿐이므로 leave-one-out 자체도 불확실하다")

## 6. 시각화 — 1UBQ (650M)
윗삼각은 실제 접촉(<8 Å), 아랫삼각은 contact head의 예측 확률이다. 대각선에서 먼 쌍이 얼마나 맞는지 본다.

In [ ]:
seq, dist = structs["1UBQ"]
head = store[("650M", "1UBQ")][2]
L = len(seq)
canvas = np.full((L, L), np.nan)
iu, il = np.triu_indices(L, k=1), np.tril_indices(L, k=-1)
canvas[iu] = (dist[iu] < CUTOFF).astype(float)
canvas[il] = head[il]
fig, ax = plt.subplots(figsize=(4.5, 4.2))
im = ax.imshow(canvas, cmap="Blues", vmin=0, vmax=1)
ax.set_title("1UBQ: true contacts (upper) / ESM-2 650M contact head (lower)", fontsize=8)
plt.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.show()

## 7. 해석 가이드와 한계
아래는 이 실행(MPS, 결정적 계산)에서 읽은 내용이다. **표를 직접 확인한다.**
- **baseline과의 비교**: 무작위 기대 정밀도는 0.03~0.08, 서열 거리 prior는 0.00~0.09다. attention 기반 점수가 **항상** 이를 넘는 것은 아니다.
  1AKI(129 aa)의 비지도 점수(sym+APC)는 8M~150M에서 0.04로 서열 거리 prior(0.08)보다 낮고 650M에서야 0.12가 된다. 1UBQ·1PGA에서는 대부분 크게 넘는다.
- **모델이 커질수록 오른다 (3개 평균 precision@L/5)**: 지도 학습된 contact head는 0.30(8M), 0.55(35M), 0.83(150M), 0.89(650M)이고, 비지도 sym+APC는 0.19, 0.31, 0.36, 0.44다.
- **보정의 효과 (비지도, 3개 평균)**: raw, 대칭화, +APC 순으로 오른다. 8M은 0.07, 0.12, 0.19, 35M은 0.15, 0.16, 0.31, 150M은 0.03, 0.23, 0.36, 650M은 0.27, 0.30, 0.44다.
  대각선(자기 attention)을 0으로 만든 뒤 APC를 적용한 변형은 일관된 차이가 없었다(−0.03~+0.01). 다만 raw는 비대칭 행렬의 윗삼각만 써서 한 방향 정보만 반영한다.
- **비지도와 지도의 차이**: 비지도 점수는 지도 학습된 contact head보다 모든 크기에서 낮다. contact head는 접촉 라벨로 학습된 선형 층이 포함되어 있어 순수 비지도 결과가 아니다†.
- **장거리 기준(서열 거리 ≥ 24)**: contact head 평균은 0.25(8M), 0.55, 0.66, 0.82(650M)다. 작은 단백질은 장거리 쌍이 적고 k = L//5가 11~25개뿐이라 통계가 불안정하다.
- **head 선택의 낙관성 (5절)**: 35M에서는 leave-one-out으로 고른 head의 정밀도(0.33/0.45/0.36)가 평가 단백질 내 최고 head(0.87/0.64/0.44)보다 낮다(낙관성 격차 평균 0.27).
  150M에서는 세 단백질 모두 같은 head (27,16)이 선택되어 격차가 0이었다. 이는 **이 세 구조에서의 관찰**이며, "일관되게 접촉을 읽는 head가 있다"고 일반화하려면 독립적인 단백질에서 추가로 확인해야 한다.

**한계**
- 단백질 3개, 모두 작고 잘 알려진 단일 도메인이다. 큰 단백질·다량체·내재적 무질서 영역에서는 다르다.
- ESM-2의 사전학습 데이터에 이 단백질 또는 상동체가 포함됐을 수 있다 (평가–사전학습 겹침, 모듈 01).
- 접촉 기준(Cβ 8 Å)과 최소 서열 거리를 바꾸면 수치가 달라진다.
- 과제: [../exercises.md](../exercises.md) 과제 4.